# TT/MPS基礎 04 — Gauge freedom と第1コアのQR分解

## このNotebookでやること

今回は **ここまで** に限定します。

1. gauge freedom の意味を数式で確認する
2. 第1ボンドに $M$ と $M^{-1}$ を入れる gauge 変換を自分で実装する
3. 変換前後でテンソルが不変か自分で確認する
4. 第1コアを QR 分解する
5. $R$ を第2コアへ吸収する処理を自分で実装する
6. $X_{\mathrm{before}}\simeq X_{\mathrm{after}}$ と $Q^TQ\simeq I$ を自分で確認する

**第2コアのQR、$L_2^TL_2=I$、mixed-canonical form にはまだ進みません。**

このNotebookは完成コードを読むためではなく、**数式を見て自分で実装するための演習用**です。

## 1. Gauge freedom

3階 TT を

$$
X(i_1,i_2,i_3)
=
\sum_{\alpha_1=1}^{r_1}
\sum_{\alpha_2=1}^{r_2}
G_1(1,i_1,\alpha_1)
G_2(\alpha_1,i_2,\alpha_2)
G_3(\alpha_2,i_3,1)
$$

とします。

第1ボンドに可逆行列

$$
M\in\mathbb{R}^{r_1\times r_1}
$$

を入れて、

$$
\widetilde G_1(1,i_1,\beta)
=
\sum_{\alpha_1}
G_1(1,i_1,\alpha_1)M(\alpha_1,\beta)
$$

$$
\widetilde G_2(\beta,i_2,\alpha_2)
=
\sum_{\gamma_1}
M^{-1}(\beta,\gamma_1)
G_2(\gamma_1,i_2,\alpha_2)
$$

と変換します。

すると

$$
\sum_{\beta}
M(\alpha_1,\beta)M^{-1}(\beta,\gamma_1)
=
\delta_{\alpha_1,\gamma_1}
$$

なので、

$$
\widetilde X=X
$$

です。

内部ボンド添字 $\alpha_1$ は物理添字ではなく、**内部ボンド空間の基底ラベル**と解釈できます。

In [1]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)


def reconstruct_tt3(G1: torch.Tensor, G2: torch.Tensor, G3: torch.Tensor) -> torch.Tensor:
    # 3個のTTコアから3階テンソルを再構成
    return torch.einsum("aib,bjc,ckd->ijk", G1, G2, G3)


def relative_fro_error(X_hat: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    # 相対Frobenius誤差
    return torch.linalg.norm(X_hat - X) / torch.linalg.norm(X)


# 小さい3階TT
n1, n2, n3 = 4, 3, 5
r1, r2 = 2, 3

G1 = torch.randn(1, n1, r1)
G2 = torch.randn(r1, n2, r2)
G3 = torch.randn(r2, n3, 1)

X = reconstruct_tt3(G1, G2, G3)

# 第1ボンドに作用させる可逆行列
M = torch.tensor([
    [2.0,  0.5],
    [-0.3, 1.4],
])
M_inv = torch.linalg.inv(M)

print("G1:", tuple(G1.shape))
print("G2:", tuple(G2.shape))
print("G3:", tuple(G3.shape))
print("X :", tuple(X.shape))


G1: (1, 4, 2)
G2: (2, 3, 3)
G3: (3, 5, 1)
X : (4, 3, 5)


## 2. 演習1 — Gauge変換を実装する

次の2式を PyTorch に落としてください。

$$
\widetilde G_1(1,i_1,\beta)
=
\sum_{\alpha_1}
G_1(1,i_1,\alpha_1)M(\alpha_1,\beta)
$$

$$
\widetilde G_2(\beta,i_2,\alpha_2)
=
\sum_{\gamma_1}
M^{-1}(\beta,\gamma_1)
G_2(\gamma_1,i_2,\alpha_2)
$$

### TODO

- `G1_gauge`
- `G2_gauge`
- `X_gauge`
- 相対誤差

を自分で実装します。

期待する結果は

$$
\frac{\|X_{\mathrm{gauge}}-X\|_F}{\|X\|_F}
\approx 0
$$

です。

### ヒント

`torch.einsum` を使う場合は、まず添字ごとに

- 古いボンド
- 新しいボンド
- 物理添字

を対応付けてから書いてください。

In [2]:
from nn_compression.tensor import mode_dot

G1_gauge = torch.tensordot(G1, M, dims=([2], [0]))
print(f"G1_guage=\n{G1_gauge}")
G1_gauge_src = mode_dot(G1, M.T, mode=2)
print(f"G1_guage_src=\n{G1_gauge_src}")


G2_gauge = torch.tensordot(M_inv,G2, dims=([1], [0]))
print(f"G2_guage=\n{G2_gauge}")
G2_gauge_src = mode_dot(G2, M_inv, mode=0)
print(f"G2_guage_src=\n{G2_gauge_src}")

X_gauge = reconstruct_tt3(G1_gauge,G2_gauge,G3)
err_gauge = relative_fro_error(X_gauge,X)

print("relative gauge error =", err_gauge.item())

print("TODO: gauge 変換を実装する")

G1_guage=
tensor([[[ 3.1700,  0.3597],
         [-4.5281, -0.2936],
         [-1.7495, -2.5003],
         [ 0.5553,  1.3749]]])
G1_guage_src=
tensor([[[ 3.1700,  0.3597],
         [-4.5281, -0.2936],
         [-1.7495, -2.5003],
         [ 0.5553,  1.3749]]])
G2_guage=
tensor([[[-0.6173,  0.0820, -0.5002],
         [ 0.7466, -0.7206, -0.0426],
         [ 0.0703, -0.5631, -0.0687]],

        [[ 0.2651,  0.1278, -0.6428],
         [-0.3567, -0.0927, -1.2566],
         [ 0.7587,  0.3464,  0.0912]]])
G2_guage_src=
tensor([[[-0.6173,  0.0820, -0.5002],
         [ 0.7466, -0.7206, -0.0426],
         [ 0.0703, -0.5631, -0.0687]],

        [[ 0.2651,  0.1278, -0.6428],
         [-0.3567, -0.0927, -1.2566],
         [ 0.7587,  0.3464,  0.0912]]])
relative gauge error = 9.953014712319636e-17
TODO: gauge 変換を実装する


## 3. QR分解による左ゲージ固定

第1コアは

$$
G_1\in\mathbb{R}^{1\times n_1\times r_1}
$$

なので、

$$
A_1
=
G_1^{\langle L\rangle}
\in
\mathbb{R}^{n_1\times r_1}
$$

と行列として見ます。

これを reduced QR 分解します。

$$
A_1=QR
$$

ここで

$$
Q\in\mathbb{R}^{n_1\times r_1},
\qquad
R\in\mathbb{R}^{r_1\times r_1}
$$

かつ

$$
Q^TQ=I_{r_1}
$$

です。

$Q$ を新しい第1コアに使い、$R$ は捨てずに第2コアへ吸収します。

$$
\widetilde G_2(\beta_1,i_2,\alpha_2)
=
\sum_{\alpha_1}
R(\beta_1,\alpha_1)
G_2(\alpha_1,i_2,\alpha_2)
$$

本質は

$$
A_1G_2=(QR)G_2=Q(RG_2)
$$

です。

## 4. 演習2 — 第1コアをQR分解して $R$ を吸収する

### TODO

1. 第1コアを $A_1$ にする
2. `torch.linalg.qr(..., mode="reduced")` で $Q,R$ を求める
3. $Q$ を $(1,n_1,r_1)$ に戻して `G1_qr` を作る
4. $R$ を第2コアの左ボンドへ作用させて `G2_qr` を作る
5. `X_qr` を再構成する

### 先に形状を確認

$$
A_1:(n_1,r_1)
$$

$$
Q:(n_1,r_1)
$$

$$
R:(r_1,r_1)
$$

$$
G_2:(r_1,n_2,r_2)
$$

特に、**$R$ が $G_2$ のどの軸に作用するか**を考えてから実装してください。

In [3]:
# TODO 2:
# 第1コアのQR分解と、Rの第2コアへの吸収を実装してください。

A1 = G1[0, :, :]
Q, R = torch.linalg.qr(A1, mode="reduced")
G1_qr = Q.unsqueeze(0)
G2_qr = torch.tensordot(R,G2, dims=([1], [0]))
X_qr = reconstruct_tt3(G1_qr,G2_qr,G3)


print("A1 shape:", A1.shape)
print("Q shape :", Q.shape)
print("R shape :", R.shape)

print("TODO: 第1コアのQR分解とRの吸収を実装する")


A1 shape: torch.Size([4, 2])
Q shape : torch.Size([4, 2])
R shape : torch.Size([2, 2])
TODO: 第1コアのQR分解とRの吸収を実装する


## 5. 演習3 — 自分で検証する

実装後、次の2点を確認します。

### 1. テンソルが保存されているか

$$
\frac{\|X_{\mathrm{qr}}-X\|_F}{\|X\|_F}
\approx 0
$$

### 2. 第1コアが左直交化されたか

$$
Q^TQ
\approx
I_{r_1}
$$

つまり、

$$
\|Q^TQ-I\|_F
$$

を計算します。

### 考える問題

一般の gauge 変換は

$$
A_1\mapsto A_1M,
\qquad
G_2\mapsto M^{-1}G_2
$$

でした。

一方、

$$
A_1=QR
$$

から

$$
Q=A_1R^{-1}
$$

です。

では、QRによる左直交化では **$M$ は何に対応するでしょうか？**

In [4]:
# TODO 3:
# 自分の実装結果を使って検証してください。
#
I = Q.T@Q
I_=Q@Q.T
I_cols = torch.eye(I.shape[1], dtype=I.dtype, device=I.device)
print(f"I:\n{I}")
print(f"I_ :, \n{I_}")
print("I shape:", I.shape)
print("I_ shape :", I_.shape)
print("||P.T - P||_F =", torch.linalg.norm(I_.T - I_).item())
print("||P @ P - P||_F =", torch.linalg.norm(I_ @ I_ - I_).item())
print("trace(P) =", torch.trace(I_).item())


err_qr = relative_fro_error(X_qr,X)
print("reconstruction_error =", err_qr.item())

orthogonality_error = relative_fro_error(I,I_cols)
print("orthogonality error  =",orthogonality_error.item())

print("TODO: 再構成誤差と直交性を検証する")


I:
tensor([[1.0000e+00, 6.9389e-17],
        [6.9389e-17, 1.0000e+00]])
I_ :, 
tensor([[ 0.3148, -0.4613, -0.0522, -0.0141],
        [-0.4613,  0.6828,  0.0048,  0.0616],
        [-0.0522,  0.0048,  0.7583, -0.4249],
        [-0.0141,  0.0616, -0.4249,  0.2441]])
I shape: torch.Size([2, 2])
I_ shape : torch.Size([4, 4])
||P.T - P||_F = 0.0
||P @ P - P||_F = 2.5215027719414395e-16
trace(P) = 2.0
reconstruction_error = 1.5258797380325605e-16
orthogonality error  = 1.716587549445839e-16
TODO: 再構成誤差と直交性を検証する


## 6. 追加演習 — 第2コアのQRと $R_2$ の局所吸収チェック

第1コアで行った操作を、第2コアにも繰り返します。

第2コア

$$
G_2\in\mathbb{R}^{r_1\times n_2\times r_2}
$$

を左展開して、

$$
A_2
=
G_2^{\langle L\rangle}
\in
\mathbb{R}^{(r_1n_2)\times r_2}
$$

とします。

これを

$$
A_2=Q_2R_2
$$

と QR 分解します。

ここで

$$
Q_2^TQ_2=I_{r_2}
$$

です。

$Q_2$ を新しい第2コアに戻し、$R_2$ は第3コアの左ボンドへ吸収します。

$$
\widetilde G_3(\beta_2,i_3,1)
=
\sum_{\alpha_2}
R_2(\beta_2,\alpha_2)
G_3(\alpha_2,i_3,1)
$$

### 今回追加する検証

全テンソルの再構成誤差だけでなく、**第2・第3コア部分だけ**を比較します。

変換前：

$$
B_{\mathrm{before}}(\alpha_1,i_2,i_3)
=
\sum_{\alpha_2}
G_2(\alpha_1,i_2,\alpha_2)
G_3(\alpha_2,i_3,1)
$$

変換後：

$$
B_{\mathrm{after}}(\alpha_1,i_2,i_3)
=
\sum_{\beta_2}
\widetilde G_2(\alpha_1,i_2,\beta_2)
\widetilde G_3(\beta_2,i_3,1)
$$

を作り、

$$
\|B_{\mathrm{after}}-B_{\mathrm{before}}\|_F
\approx 0
$$

を確認します。

これにより、

> $R_2$ を第3コアへ正しく吸収した結果、局所的な第2–第3コア収縮が保存されている

ことを直接確認できます。

In [5]:
# TODO 4:
# 第2コアを左展開し、QR 分解してください。

A2 = G2_qr.reshape(r1*n2,r2)
Q2, R2 = torch.linalg.qr(A2, mode="reduced")
#
# Q2 を G2 と同じ3階形状に戻す
G2_qr2 = Q2.reshape(r1,n2,r2)
#
# R2 を G3 の左ボンドへ吸収する
G3_qr2 = torch.tensordot(R2,G3, dims=([1], [0]))
#
B_bef=torch.tensordot(G2_qr,G3, dims=([2], [0]))
B_aft=torch.tensordot(G2_qr2,G3_qr2, dims=([2], [0]))
B_def_abs=torch.norm(B_aft-B_bef).item()

print("A2 shape:", A2.shape)
print("Q2 shape:", Q2.shape)
print("R2 shape:", R2.shape)
print("B_def_abs:", B_def_abs)

print("TODO: 第2コアのQR分解とR2の第3コアへの吸収を実装する")


A2 shape: torch.Size([6, 3])
Q2 shape: torch.Size([6, 3])
R2 shape: torch.Size([3, 3])
B_def_abs: 5.5558315673428316e-15
TODO: 第2コアのQR分解とR2の第3コアへの吸収を実装する


### 追加検証 — 第2・第3コアの局所収縮が保存されるか

ここでは全テンソルを再構成する前に、$G_2$ と $G_3$ だけを収縮して確認します。

確認したいのは

$$
B_{\mathrm{before}}
\simeq
B_{\mathrm{after}}
$$

です。

### TODO

1. QR前の `G2_qr, G3` から `B_before` を作る
2. QR後の `G2_qr2, G3_qr2` から `B_after` を作る
3. 次を計算する

$$
\|B_{\mathrm{after}}-B_{\mathrm{before}}\|_F
$$

この値が丸め誤差程度なら、$R_2$ の吸収は局所的にも正しいです。

### さらに確認

同時に、

$$
\|Q_2^TQ_2-I\|_F
$$

も計算して、第2コアが左直交化されたことを確認してください。

In [6]:
# TODO 5:
# 第2コアのQR前後で、第2-第3コア部分の局所収縮が保存されるか確認してください。
#
B_bef=torch.tensordot(G2_qr,G3, dims=([2], [0]))
B_aft=torch.tensordot(G2_qr2,G3_qr2, dims=([2], [0]))
B_def_abs=torch.norm(B_aft-B_bef).item()
print("B_def_abs:", B_def_abs)
print("A2 shape:", A2.shape)
#
I2 = Q2.T@Q2
I_cols2 = torch.eye(Q2.shape[1], dtype=Q2.dtype, device=Q2.device)
orthogonality_error_q2 = torch.norm(I_cols2-I2).item()
print("Q2 orthogonality error       =",orthogonality_error_q2)

print("TODO: R2の局所吸収とQ2の直交性を検証する")


B_def_abs: 5.5558315673428316e-15
A2 shape: torch.Size([6, 3])
Q2 orthogonality error       = 4.660953740672398e-16
TODO: R2の局所吸収とQ2の直交性を検証する


## 7. 今回の到達点

今回のNotebookでは、

$$
\text{gauge freedom}
\rightarrow
\text{第1コアのQR}
\rightarrow
\text{$R_1$を第2コアへ吸収}
\rightarrow
\text{第2コアのQR}
\rightarrow
\text{$R_2$を第3コアへ吸収}
$$

までを演習します。

特に今回は、全体の再構成誤差だけでなく、

$$
\|B_{\mathrm{after}}-B_{\mathrm{before}}\|_F
\approx 0
$$

を確認し、**$R_2$ の第3コアへの吸収が局所的にも正しい**ことを検証します。

自分の実装で最終的に

$$
X_{\mathrm{before}}
\simeq
X_{\mathrm{after}}
$$

$$
Q_1^TQ_1
\simeq
I
$$

$$
Q_2^TQ_2
\simeq
I
$$

を確認できれば、この段階は完了です。

次は、

- 3階TT全体の left-canonical form
- $L_2^TL_2=I$
- mixed-canonical form

とのつながりを整理します。

## 8. 追加確認 — なぜ $L_2^T L_2 = I_{r_2}$ が必然的に成り立つのか

ここでは**新しい操作はしません**。

すでに第1・第2コアを QR 分解した結果、

$$
Q^TQ=I_{r_1}
$$

$$
Q_2^TQ_2=I_{r_2}
$$

が成り立っています。

この2つから、左ブロック

$$
L_2(i_1,i_2,\alpha_2)
=
\sum_{\alpha_1}
G_1^{\mathrm{qr}}(1,i_1,\alpha_1)
G_2^{\mathrm{qr2}}(\alpha_1,i_2,\alpha_2)
$$

が

$$
L_2^TL_2=I_{r_2}
$$

を満たすことを確認します。

---

$L_2^TL_2$ の $(\alpha_2,\beta_2)$ 成分は

$$
(L_2^TL_2)_{\alpha_2,\beta_2}
=
\sum_{i_1,i_2}
L_2(i_1,i_2,\alpha_2)
L_2(i_1,i_2,\beta_2)
$$

です。

ここへ $L_2$ の定義を代入すると、

$$
\begin{aligned}
(L_2^TL_2)_{\alpha_2,\beta_2}
&=
\sum_{i_1,i_2}
\sum_{\alpha_1,\gamma_1}
G_1^{\mathrm{qr}}(1,i_1,\alpha_1)
G_2^{\mathrm{qr2}}(\alpha_1,i_2,\alpha_2) \\
&\qquad\qquad\times
G_1^{\mathrm{qr}}(1,i_1,\gamma_1)
G_2^{\mathrm{qr2}}(\gamma_1,i_2,\beta_2).
\end{aligned}
$$

第1コアは QR 後なので、

$$
\sum_{i_1}
G_1^{\mathrm{qr}}(1,i_1,\alpha_1)
G_1^{\mathrm{qr}}(1,i_1,\gamma_1)
=
\delta_{\alpha_1,\gamma_1}.
$$

したがって、

$$
\begin{aligned}
(L_2^TL_2)_{\alpha_2,\beta_2}
&=
\sum_{i_2}
\sum_{\alpha_1,\gamma_1}
\delta_{\alpha_1,\gamma_1}
G_2^{\mathrm{qr2}}(\alpha_1,i_2,\alpha_2)
G_2^{\mathrm{qr2}}(\gamma_1,i_2,\beta_2) \\
&=
\sum_{\alpha_1,i_2}
G_2^{\mathrm{qr2}}(\alpha_1,i_2,\alpha_2)
G_2^{\mathrm{qr2}}(\alpha_1,i_2,\beta_2).
\end{aligned}
$$

一方、第2コアを left unfolding した行列は QR 後の $Q_2$ なので、

$$
Q_2^TQ_2=I_{r_2}.
$$

その $(\alpha_2,\beta_2)$ 成分は

$$
\sum_{\alpha_1,i_2}
G_2^{\mathrm{qr2}}(\alpha_1,i_2,\alpha_2)
G_2^{\mathrm{qr2}}(\alpha_1,i_2,\beta_2)
=
\delta_{\alpha_2,\beta_2}.
$$

したがって、

$$
\boxed{
L_2^TL_2=I_{r_2}
}
$$

です。

つまり $L_2^TL_2=I$ は新しく課した条件ではなく、**第1・第2コアを左から順に直交化した結果として、左ブロック全体に自動的に引き継がれる直交性**です。

---

### 自分で確認すること

上の数式をコードにつなげます。

1. `G1_qr` と `G2_qr2` を内部ボンドで収縮して $L_2$ を作る
2. $(i_1,i_2)$ を1つの行添字にまとめる
3. $L_2^TL_2$ を計算する
4. 単位行列 $I_{r_2}$ と比較する
5. 次を確認する

$$
\|L_2^TL_2-I_{r_2}\|_F
\approx 0
$$


In [7]:
# TODO 6:
# 上の導出を、そのままコードで確認してください。
#
# 1. G1_qr と G2_qr2 を内部ボンドで収縮して L2_tensor を作る
L2_tensor = torch.tensordot(G1_qr, G2_qr2, dims=([2], [0]))

# 2. 左端ランク r0=1 を落とし、
#    (i1, i2) を1つの行添字にまとめて L2 を作る
L2 = L2_tensor.reshape(n1 * n2, r2)
#
# 3. L2^T L2 を計算する
L2_gram = L2.T @ L2
#
# 4. I_{r2} を作る
I_r2 = torch.eye(L2.shape[1], dtype=L2.dtype, device=L2.device)
#
# 5. ||L2^T L2 - I||_F を計算する
L2_orthogonality_error = torch.linalg.matrix_norm(L2_gram - I_r2, ord="fro").item()

print("L2_tensor shape:", tuple(L2_tensor.shape))
print("L2 shape:", tuple(L2.shape))
print("L2^T L2 =")
print(L2_gram)
print("||L2^T L2 - I||_F =", L2_orthogonality_error)


L2_tensor shape: (1, 4, 3, 3)
L2 shape: (12, 3)
L2^T L2 =
tensor([[ 1.0000e+00, -5.5511e-17,  3.4694e-17],
        [-5.5511e-17,  1.0000e+00, -1.8041e-16],
        [ 3.4694e-17, -1.8041e-16,  1.0000e+00]])
||L2^T L2 - I||_F = 7.19305970752107e-16
